<a href="https://colab.research.google.com/github/Nubiafernanda/Flahcard2c/blob/main/Cripografia_cofre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Atividede 1


Objetivos do Cofre Digital

• Proteger informações pessoais: Impede que terceiros leiam senhas do Wi-Fi, e-mail ou banco caso acessem o computador ou o arquivo meu_cofre_seguro.txt.
• Garantir acessibilidade visual: Usa letras grandes, alto contraste e botões coloridos para facilitar a leitura e evitar cliques errados de idosos ou iniciantes.
• Educar contra golpes: Exibe um quadro de alerta fixo no topo da tela para ensinar boas práticas de segurança digital antes mesmo de o usuário digitar seus dados.
• Preservar a privacidade na tela: Oculta a senha mestra com bolinhas e limpa o campo de texto automaticamente após salvar, evitando que alguém veja os dados por cima do ombro.

Como o Usuário Vai Usar (Passo a Passo)


1. Para Guardar Novas Informações

1. Leia o aviso amarelo: O usuário confere as dicas rápidas no topo para lembrar de nunca compartilhar senhas e anotar a senha mestra em um papel seguro.
2. Crie ou digite a Senha Mestra: No campo 1, o usuário digita sua chave principal. Se tiver dúvida do que digitou, pode clicar no botão 👁️ Mostrar para conferir e depois clicar em 🙈 Ocultar.
3. Escreva os segredos: No campo 2, o usuário digita as senhas ou anotações que deseja proteger.
4. Tranque o cofre: O usuário clica no botão verde 🔒 Trancar e Salvar. O sistema criptografa tudo, salva no arquivo .txt, limpa o campo de texto e mostra uma mensagem verde de sucesso.

2. Para Ler as Informações Guardadas

1. Informe a chave: O usuário digita a mesma Senha Mestra usada anteriormente no campo 1.
2. Abra o cofre: O usuário clica no botão azul 🔓 Abrir Cofre. Se a senha estiver certa, o texto original aparece na caixa cinza inferior. Se estiver errada, o sistema bloqueia a leitura e avisa que a senha está incorreta.

3. Para Encerrar o Uso com Segurança

• Limpe a tela: Assim que terminar de ler suas senhas, o usuário clica no botão cinza 🧹 Limpar Tela (Esconder tudo). Isso apaga imediatamente a senha digitada e o conteúdo aberto da tela.
Se você estiver preparando um manual ou apresentação sobre esse projeto para seus familiares, posso ajudar a criar:
• Um guia ilustrado passo a passo em texto simples para imprimir e deixar ao lado do computador.
• Uma explicação simplificada da criptografia (SHA-256 + XOR + Base64) usando analogias do dia a dia.


In [3]:
import hashlib
import base64
import os
from IPython.display import display, HTML
from google.colab import output

ARQUIVO_COFRE = "meu_cofre_seguro.txt"

# ==========================================
# 1. BACK-END EM PYTHON (CRIPTOGRAFIA E ARQUIVO)
# ==========================================

def derivar_chave(senha_mestra):
    """Gera um hash SHA-256 de 32 bytes a partir da Senha Mestra."""
    return hashlib.sha256(senha_mestra.encode('utf-8')).digest()

def processar_xor(dados_bytes, chave_bytes):
    """Aplica a operação XOR byte a byte repetindo a chave SHA-256."""
    return bytes([b ^ chave_bytes[i % len(chave_bytes)] for i, b in enumerate(dados_bytes)])

def salvar_dados_criptografados(senha_mestra, texto_segredo):
    if not senha_mestra or not texto_segredo:
        return "⚠️ Por favor, preencha a Senha Mestra e o texto que deseja guardar."

    try:
        chave = derivar_chave(senha_mestra)
        dados_bytes = texto_segredo.encode('utf-8')

        # Criptografa com XOR e codifica em Base64 para salvar como texto limpo
        cifrado_xor = processar_xor(dados_bytes, chave)
        texto_base64 = base64.b64encode(cifrado_xor).decode('utf-8')

        with open(ARQUIVO_COFRE, "w", encoding="utf-8") as f:
            f.write(texto_base64)

        return f"✅ Sucesso! Seus dados foram trancados e salvos no arquivo '{ARQUIVO_COFRE}'."
    except Exception as e:
        return f"❌ Erro ao salvar: {str(e)}"

def ler_dados_descriptografados(senha_mestra):
    if not senha_mestra:
        return "⚠️ Digite sua Senha Mestra para abrir o cofre."

    if not os.path.exists(ARQUIVO_COFRE):
        return "ℹ️ Nenhum cofre encontrado ainda. Salve alguma informação primeiro."

    try:
        with open(ARQUIVO_COFRE, "r", encoding="utf-8") as f:
            texto_base64 = f.read().strip()

        chave = derivar_chave(senha_mestra)
        cifrado_xor = base64.b64decode(texto_base64)

        # Reverte o XOR usando a mesma chave
        dados_bytes = processar_xor(cifrado_xor, chave)
        texto_original = dados_bytes.decode('utf-8')

        return f"🔓 COFRE ABERTO:\n\n{texto_original}"
    except UnicodeDecodeError:
        return "🚫 SENHA INCORRETA! Os dados não puderam ser lidos. Verifique se digitou a senha certa."
    except Exception as e:
        return f"❌ Erro ao ler o arquivo: {str(e)}"

# Registra as funções Python para serem chamadas pelo JavaScript
output.register_callback('notebook.salvar_cofre', salvar_dados_criptografados)
output.register_callback('notebook.abrir_cofre', ler_dados_descriptografados)

# ==========================================
# 2. FRONT-END EM HTML, CSS E JAVASCRIPT
# ==========================================

interface_html = """
<!DOCTYPE html>
<html lang="pt-BR">
<head>
<style>
  .cofre-container {
    font-family: 'Segoe UI', Arial, sans-serif;
    max-width: 680px;
    margin: 15px auto;
    background-color: #f8fafc;
    border: 3px solid #cbd5e1;
    border-radius: 16px;
    padding: 25px;
    color: #1e293b;
    box-shadow: 0 10px 25px rgba(0,0,0,0.1);
  }
  .cofre-titulo {
    text-align: center;
    color: #0f172a;
    font-size: 28px;
    margin-top: 0;
    margin-bottom: 20px;
  }
  /* Aviso Educativo Antigolpes */
  .aviso-golpes {
    background-color: #fef3c7;
    border-left: 8px solid #d97706;
    padding: 16px 20px;
    border-radius: 10px;
    margin-bottom: 25px;
    font-size: 18px;
    line-height: 1.5;
  }
  .aviso-golpes strong {
    color: #92400e;
    font-size: 20px;
    display: block;
    margin-bottom: 8px;
  }
  .aviso-golpes ul {
    margin: 5px 0 0 0;
    padding-left: 22px;
  }
  .aviso-golpes li {
    margin-bottom: 6px;
  }
  /* Campos de Entrada */
  .campo-grupo {
    margin-bottom: 20px;
  }
  label {
    display: block;
    font-size: 20px;
    font-weight: bold;
    margin-bottom: 8px;
    color: #334155;
  }
  .senha-wrapper {
    display: flex;
    gap: 10px;
  }
  input[type="password"], input[type="text"], textarea {
    width: 100%;
    padding: 14px;
    font-size: 20px;
    border: 2px solid #94a3b8;
    border-radius: 10px;
    box-sizing: border-box;
    background-color: #ffffff;
    color: #0f172a;
  }
  input:focus, textarea:focus {
    outline: none;
    border-color: #2563eb;
    box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.2);
  }
  /* Botões Grandes e Coloridos */
  .btn-olho {
    background-color: #e2e8f0;
    color: #1e293b;
    border: 2px solid #94a3b8;
    border-radius: 10px;
    padding: 0 18px;
    font-size: 18px;
    font-weight: bold;
    cursor: pointer;
    white-space: nowrap;
  }
  .botoes-acao {
    display: flex;
    gap: 15px;
    margin-top: 10px;
  }
  .btn {
    flex: 1;
    padding: 18px;
    font-size: 21px;
    font-weight: bold;
    border: none;
    border-radius: 12px;
    cursor: pointer;
    color: white;
    transition: transform 0.1s, opacity 0.2s;
  }
  .btn:active {
    transform: scale(0.98);
  }
  .btn-salvar {
    background-color: #16a34a; /* Verde */
  }
  .btn-salvar:hover {
    background-color: #15803d;
  }
  .btn-abrir {
    background-color: #2563eb; /* Azul */
  }
  .btn-abrir:hover {
    background-color: #1d4ed8;
  }
  .btn-limpar {
    background-color: #64748b; /* Cinza */
    margin-top: 12px;
    width: 100%;
    padding: 12px;
    font-size: 18px;
  }
  /* Área de Resposta */
  .caixa-resultado {
    margin-top: 25px;
    padding: 18px;
    background-color: #e2e8f0;
    border-radius: 10px;
    font-size: 20px;
    font-weight: 500;
    white-space: pre-wrap;
    min-height: 60px;
    border: 2px solid #cbd5e1;
  }
</style>
</head>
<body>

<div class="cofre-container">
  <h2 class="cofre-titulo">🔐 Meu Cofre Digital Seguro</h2>

  <!-- Aviso Educativo -->
  <div class="aviso-golpes">
    <strong>🛡️ Dicas de Ouro Contra Golpes na Internet:</strong>
    <ul>
      <li><b>Ninguém do banco pede sua senha:</b> Gerentes ou centrais nunca ligam pedindo senhas, códigos por SMS ou transferências de teste.</li>
      <li><b>Desconfie de urgência:</b> Golpistas dizem que sua conta será bloqueada hoje para fazer você agir sem pensar.</li>
      <li><b>Anote sua Senha Mestra no papel:</b> Guarde-a em uma gaveta segura em casa. Se esquecê-la, ninguém consegue recuperar o cofre.</li>
    </ul>
  </div>

  <!-- Campo da Senha Mestra -->
  <div class="campo-grupo">
    <label for="senhaMestra">1. Digite sua Senha Mestra (a chave do cofre):</label>
    <div class="senha-wrapper">
      <input type="password" id="senhaMestra" placeholder="Ex: CasaAzul2026!">
      <button type="button" class="btn-olho" onclick="alternarSenha()">👁️ Mostrar</button>
    </div>
  </div>

  <!-- Campo de Dados -->
  <div class="campo-grupo">
    <label for="dadosSecretos">2. Escreva o que deseja guardar em segredo:</label>
    <textarea id="dadosSecretos" rows="4" placeholder="Ex: Senha do Wi-Fi: familia123&#10;Senha do E-mail: sol456..."></textarea>
  </div>

  <!-- Botões Principais -->
  <div class="botoes-acao">
    <button class="btn btn-salvar" onclick="acaoSalvar()">🔒 Trancar e Salvar</button>
    <button class="btn btn-abrir" onclick="acaoAbrir()">🔓 Abrir Cofre</button>
  </div>

  <button class="btn btn-limpar" onclick="limparTela()">🧹 Limpar Tela (Esconder tudo)</button>

  <!-- Resultado -->
  <div id="resultado" class="caixa-resultado">O cofre está fechado. Digite sua senha para salvar ou ler seus dados.</div>
</div>

<script>
  function alternarSenha() {
    const campo = document.getElementById('senhaMestra');
    const botao = document.querySelector('.btn-olho');
    if (campo.type === 'password') {
      campo.type = 'text';
      botao.innerText = '🙈 Ocultar';
    } else {
      campo.type = 'password';
      botao.innerText = '👁️ Mostrar';
    }
  }

  async function acaoSalvar() {
    const senha = document.getElementById('senhaMestra').value;
    const dados = document.getElementById('dadosSecretos').value;
    const resDiv = document.getElementById('resultado');

    resDiv.innerText = "⏳ Trancando dados com segurança...";

    // Chama a função Python no Colab
    const resposta = await google.colab.kernel.invokeFunction(
      'notebook.salvar_cofre', [senha, dados], {}
    );

    resDiv.innerText = resposta.data['text/plain'].replace(/^'|'$/g, '').replace(/\\n/g, '\n');
    document.getElementById('dadosSecretos').value = ""; // Limpa o texto por segurança
  }

  async function acaoAbrir() {
    const senha = document.getElementById('senhaMestra').value;
    const resDiv = document.getElementById('resultado');

    resDiv.innerText = "⏳ Verificando chave...";

    // Chama a função Python no Colab
    const resposta = await google.colab.kernel.invokeFunction(
      'notebook.abrir_cofre', [senha], {}
    );

    resDiv.innerText = resposta.data['text/plain'].replace(/^'|'$/g, '').replace(/\\n/g, '\n');
  }

  function limparTela() {
    document.getElementById('senhaMestra').value = "";
    document.getElementById('dadosSecretos').value = "";
    document.getElementById('resultado').innerText = "Tela limpa! O cofre está fechado.";
  }
</script>

</body>
</html>
"""

display(HTML(interface_html))
